In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ouro")
spark.sql("SELECT * FROM prata.fato_despesas").createOrReplaceTempView("prata_despesas_view")
print("Pronto.")

# Entrega 1: Score de Anomalia (Z-Score)

## O que é
O Z-Score mede o quão distante um valor está da média do seu grupo,
considerando também a variação natural (desvio padrão) desse grupo.

## Fórmula
Z-Score = (valor_do_deputado - media_do_grupo) / desvio_padrao_do_grupo

## Como interpretar
- Z-Score = 0: gasto exatamente na média
- Z-Score entre -2 e +2: gasto normal (95% dos casos)
- Z-Score acima de 3: anomalia (gasto muito acima do esperado)
- Z-Score acima de 10: fortíssimo indicativo de valor atípico

## Grupos de comparação
A média e o desvio padrão são calculados separadamente para cada
combinação de:
- Categoria de despesa (ex: COMBUSTÍVEIS, HOSPEDAGEM, etc.)
- Estado do deputado (sigla_uf)

Isso evita comparar realidades diferentes. Exemplo:
- Combustível no RS tem média de R$ 215
- Combustível em SP pode ter média de R$ 350
- Hospedagem tem valores naturalmente maiores que combustível
- Cada grupo tem sua própria média e seu próprio desvio

## Cuidados técnicos
- NULLIF(e.desvio, 0): evita erro de divisão por zero quando
  todos do grupo têm o mesmo valor
- COALESCE(e.desvio, 0): se não houver desvio calculado, assume 0

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.anomalias_despesas
USING DELTA
COMMENT 'Camada Ouro - Score de anomalia (Z-Score) por categoria e UF'
AS
WITH estatisticas AS (
    SELECT 
        categoria,
        sigla_uf,
        AVG(valor_liquido) AS media,
        STDDEV(valor_liquido) AS desvio
    FROM prata_despesas_view
    GROUP BY categoria, sigla_uf
)
SELECT 
    d.id_deputado,
    d.nome_deputado,
    d.sigla_partido,
    d.sigla_uf,
    d.categoria,
    d.data_documento,
    d.nome_fornecedor,
    d.cnpj_cpf_fornecedor,
    d.valor_liquido,
    ROUND(e.media, 2) AS media_categoria_uf,
    ROUND(COALESCE(e.desvio, 0), 2) AS desvio_categoria_uf,
    ROUND((d.valor_liquido - e.media) / NULLIF(e.desvio, 0), 2) AS z_score
FROM prata_despesas_view d
JOIN estatisticas e ON d.categoria = e.categoria AND d.sigla_uf = e.sigla_uf

In [0]:
%sql
-- Top 10 anomalias (maiores Z-Scores)
SELECT 
    nome_deputado,
    sigla_uf,
    categoria,
    nome_fornecedor,
    valor_liquido,
    media_categoria_uf,
    z_score
FROM ouro.anomalias_despesas
ORDER BY z_score DESC
LIMIT 10

# Entrega 2: Ranking de Fornecedores
Top fornecedores mais pagos com valor total e quantidade de deputados atendidos.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.ranking_fornecedores
USING DELTA
COMMENT 'Camada Ouro - Ranking de fornecedores mais pagos'
AS
SELECT 
    nome_fornecedor,
    cnpj_cpf_fornecedor,
    COUNT(*) AS qtd_despesas,
    COUNT(DISTINCT id_deputado) AS qtd_deputados,
    ROUND(SUM(valor_liquido), 2) AS total_pago,
    ROUND(AVG(valor_liquido), 2) AS ticket_medio
FROM prata_despesas_view
GROUP BY nome_fornecedor, cnpj_cpf_fornecedor
ORDER BY total_pago DESC

In [0]:
%sql
SELECT 
    nome_fornecedor,
    cnpj_cpf_fornecedor,
    qtd_deputados,
    total_pago,
    ticket_medio
FROM ouro.ranking_fornecedores
LIMIT 10

# Entrega 3: Top 10 Gastos por Partido
Relatório mensal com os 10 maiores gastos por partido.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.top10_gastos_partido
USING DELTA
COMMENT 'Camada Ouro - Top 10 gastos mensais por partido'
AS
WITH ranking AS (
    SELECT 
        sigla_partido,
        ano,
        mes,
        nome_deputado,
        nome_fornecedor,
        categoria,
        valor_liquido,
        ROW_NUMBER() OVER (PARTITION BY sigla_partido, ano, mes ORDER BY valor_liquido DESC) AS posicao
    FROM prata_despesas_view
)
SELECT 
    sigla_partido,
    ano,
    mes,
    posicao,
    nome_deputado,
    nome_fornecedor,
    categoria,
    valor_liquido
FROM ranking
WHERE posicao <= 10

In [0]:
%sql
-- Top 10 gastos do PT em janeiro de 2024
SELECT 
    ano,
    mes,
    sigla_partido,
    posicao,
    nome_deputado,
    nome_fornecedor,
    categoria,
    valor_liquido
FROM ouro.top10_gastos_partido
WHERE sigla_partido = 'PT' AND ano = 2024 AND mes = 1
ORDER BY posicao

# Entrega 4: Total de Gastos por Partido (consolidado)
Visao agregada para relatorio gerencial.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.gastos_por_partido
USING DELTA
COMMENT 'Camada Ouro - Consolidado de gastos por partido'
AS
SELECT 
    sigla_partido,
    ano,
    COUNT(*) AS qtd_despesas,
    COUNT(DISTINCT id_deputado) AS qtd_deputados,
    ROUND(SUM(valor_liquido), 2) AS total_gasto,
    ROUND(AVG(valor_liquido), 2) AS media_despesa,
    ROUND(SUM(valor_liquido) / COUNT(DISTINCT id_deputado), 2) AS gasto_por_deputado
FROM prata_despesas_view
GROUP BY sigla_partido, ano
ORDER BY total_gasto DESC

In [0]:
%sql
SELECT 
    sigla_partido,
    qtd_deputados,
    total_gasto,
    gasto_por_deputado
FROM ouro.gastos_por_partido
WHERE ano = 2024
ORDER BY total_gasto DESC
LIMIT 10

# Pendência: Automação do Relatório Mensal

## Requisito do desafio
Relatório mensal automatizado com top 10 gastos por partido.

## Solução atual
A view `ouro.relatorio_mensal_partido` está pronta e funcional. Qualquer consulta
com filtro de ano e mês retorna o top 10 de cada partido.

Exemplo de consulta:
SELECT * FROM ouro.relatorio_mensal_partido WHERE ano = 2024 AND mes = 1

## Limitação do ambiente
O Databricks Community Edition não oferece suporte a jobs agendados (Workflows).
Em um ambiente completo (Standard ou Premium), a automação seria implementada assim:

### Job proposto
- Nome: Relatorio Mensal CEAP - Top 10 por Partido
- Trigger: agendamento mensal, dia 5 de cada mês
- Tarefa: executar notebook que consulta a view do mês anterior
- Saída: exportar resultado como CSV para bucket S3 ou enviar por email
- Monitoramento: alerta via webhook se execução falhar

### Código de notebook para o job (simulação)
```python
# Parametro: mes e ano a serem consultados
mes_ref = dbutils.widgets.get("mes")
ano_ref = dbutils.widgets.get("ano")

df = spark.sql(f"""
    SELECT * FROM ouro.relatorio_mensal_partido
    WHERE ano = {ano_ref} AND mes = {mes_ref}
    ORDER BY sigla_partido, posicao
""")

df.show(1000, truncate=False)
# Em producao: df.write.csv(f"dbfs:/relatorios/ceap/{ano_ref}_{mes_ref}.csv")